# Extend the 260723 tournament entrants from 3e6 to 4e6 steps

The eight shortlisted `260723` models all stopped at exactly 3e6 steps. Self-play ELO was still climbing in their final million steps (7 of 8 with a positive slope over 2–3e6) and policy entropy was still falling, so 3e6 is short of convergence. This resumes each of them to 4e6.

This uses `resume_training_runs` in `trainer_and_simulator_functions.py`, not `sequential_runs`. `sequential_runs` trains a contiguous block of fresh models, so it passes `--resume` to the first model only and every later run starts clean. Here all eight runs already exist and need resuming, and the ids are a hand-picked non-contiguous set.

**Two things have to line up, or mlagents does nothing and reports success:**
1. `max_steps` in the config must exceed the step the runs stopped at. The template says `3e06`, so resuming against it would exit immediately with nothing to train. The function writes a separate `SocialConfig_4e06.yaml` rather than editing the shared template, which future fresh batches still use.
2. The run id must locate the run under `<run_dir>/results/`. These runs were filed into `results/2607/260723/`, and mlagents resolves output as `os.path.join(results_dir, run_id)`, so the id has to be `2607/260723/260723_0034` — the bare folder name would create a new empty run. The function derives this prefix itself.

Both are checked before anything launches, so a mistake costs a message rather than a night of no-op runs.

> **The tournament is insulated from this.** `run_tournament_260723.ipynb` points at `results/2607/260723_frozen_3e6`, a snapshot of the eight entrants taken before this extension. Resuming here overwrites `checkpoint.pt` in the live run dirs, which the tournament re-reads for every matchup, so without the snapshot the round-robin would mix 3e6 and 4e6 policies.

In [5]:
import importlib
import sys
from pathlib import Path

REPO_ROOT = Path("~/repos/agent_training").expanduser()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import trainer_and_simulator_functions as tsf
importlib.reload(tsf)

<module 'trainer_and_simulator_functions' from '/home/tom/repos/agent_training/trainer_and_simulator_functions.py'>

In [6]:
# The eight tournament entrants (see model_characterisation_demo260723 Stage 4).
RUN_IDS = [
    "260723_0034",  # central
    "260723_0039",  # central
    "260723_0044",  # intermediate
    "260723_0035",  # peripheral
    "260723_0001",  # peripheral
    "260723_0033",  # peripheral
    "260723_0051",  # peripheral
    "260723_0023",  # peripheral
]

RESULTS_ROOT = Path("~/Unity/Octagon/results/2607/260723").expanduser()   # the LIVE runs
RUN_DIR      = Path("~/Unity/Octagon").expanduser()                       # parent of results/
IN_YAML      = Path("~/Unity/Octagon/Assets/Scripts/MLConfigFiles/SocialConfig.yaml").expanduser()
UNITY_BUILD  = Path("~/Unity/Octagon/builds/260723/build_260723_01_social.x86_64").expanduser()

# Written beside the repo's other configs; the shared template is left untouched.
EXTENDED_YAML = REPO_ROOT / "configs" / "SocialConfig_4e06.yaml"

MAX_STEPS = 4e6

# n_envs stays at the 16 these runs were trained with. Changing the number of parallel
# environments changes how experience is collected per update, so a resume at a different
# value is not a clean continuation.
N_ENVS = 16

# Training ports are 5005 + 20*i -> 5005..5145 for 8 runs. The tournament uses 9100..10360,
# so the two do not collide even if they run at the same time.
BASE_PORT = 5005

## Dry run first

Prints where each run stands, how many steps it will gain, and the mlagents run id it will resume, without launching anything. Check the run ids look like `2607/260723/260723_00XX` before going further.

In [7]:
starts = tsf.resume_training_runs(
    run_ids=RUN_IDS,
    results_root=RESULTS_ROOT,
    run_dir=RUN_DIR,
    in_yaml=IN_YAML,
    unity_build=UNITY_BUILD,
    max_steps=MAX_STEPS,
    extended_yaml=EXTENDED_YAML,
    n_envs=N_ENVS,
    base_port=BASE_PORT,
    dry_run=True,
)

ValueError: 260723_0034 is already at 4000042 steps, at or past max_steps 4000000. It would resume and exit with nothing to train.

## Launch

Sequential, one model at a time. From the 260723 checkpoint mtimes a full 3e6 run took ~2h40m, so an extra 1e6 steps is **~55 min per model, ~7.5 h for all eight** — longer if the tournament is running alongside (expect roughly +30%, and the tournament slows by about the same).

Resuming writes into the **live** run dirs, so afterwards `results/2607/260723/` holds 4e6 policies while `260723_frozen_3e6/` still holds the 3e6 ones. That pairing is what makes the "is peripheral undertrained or just weaker?" comparison possible: the same eight models at two training lengths.

In [ ]:
reached = tsf.resume_training_runs(
    run_ids=RUN_IDS,
    results_root=RESULTS_ROOT,
    run_dir=RUN_DIR,
    in_yaml=IN_YAML,
    unity_build=UNITY_BUILD,
    max_steps=MAX_STEPS,
    extended_yaml=EXTENDED_YAML,
    n_envs=N_ENVS,
    base_port=BASE_PORT,
    seed=17,
    dry_run=False,
)

extended config written: /home/tom/repos/agent_training/configs/SocialConfig_4e06.yaml  (max_steps -> 4000000)

Resuming 8 runs under /home/tom/Unity/Octagon/results/2607/260723
  260723_0034: 3,000,063 ->   4,000,000  (+999,937 steps)  run-id '2607/260723/260723_0034'
  260723_0039: 3,000,117 ->   4,000,000  (+999,883 steps)  run-id '2607/260723/260723_0039'
  260723_0044: 3,000,101 ->   4,000,000  (+999,899 steps)  run-id '2607/260723/260723_0044'
  260723_0035: 3,000,112 ->   4,000,000  (+999,888 steps)  run-id '2607/260723/260723_0035'
  260723_0001: 3,000,074 ->   4,000,000  (+999,926 steps)  run-id '2607/260723/260723_0001'
  260723_0033: 3,000,114 ->   4,000,000  (+999,886 steps)  run-id '2607/260723/260723_0033'
  260723_0051: 3,000,009 ->   4,000,000  (+999,991 steps)  run-id '2607/260723/260723_0051'
  260723_0023: 3,000,036 ->   4,000,000  (+999,964 steps)  run-id '2607/260723/260723_0023'

=== [1/8] Resuming 260723_0034 (from 3,000,063 steps) as run-id '2607/260723/260723_0

## After it finishes

- Take a `260723_frozen_4e6` snapshot the same way, so a later 4e6 tournament has immutable entrants too.
- Re-check the ELO slopes over 3–4e6. If they have flattened, 4e6 is enough; if peripheral models are still climbing where central ones have levelled, that is the direct evidence for the undertraining hypothesis.
- The cheapest test of that hypothesis is a small side tournament: each extended model against its own frozen 3e6 self. A model that reliably beats its earlier self gained something real from the extra million steps.

## Extend the rest of the batch

The eight entrants are done. This extends every other run in `results/2607/260723` to the
same 4e6, so the whole batch is at one training length and the characterisation notebook can
be re-run without a step-count confound.

`runs_below_max_steps` picks the runs still under the ceiling, which keeps the eight finished
ones out. `resume_training_runs` raises on a run already at max_steps, so the filter is what
lets a part-finished batch be resumed. Both read the checkpoints, so this is safe to re-run
after an interruption: it returns only what is still short.

**Scale.** 47 runs, 47.1M steps in total, about 55 min each, so roughly **43 hours**. Ports
run 5005 to 5941, clear of the tournament's 9100+ and the rerun range at 10500+, so this can
run alongside the tournament. Expect both to slow by about 30% if you do.

`260723_0055` is the one irregular run: it stopped at 2,899,887 rather than 3e6, so it gains
1.1e6 steps instead of 1e6. It reaches the same 4e6 ceiling as the rest.

In [8]:
# Every run in the folder that is still below MAX_STEPS. The eight entrants are already at
# 4e6, so they drop out here rather than tripping the guard in resume_training_runs.
REST = tsf.runs_below_max_steps(
    results_root=RESULTS_ROOT,
    max_steps=MAX_STEPS,
    behaviour_name="OctagonAgentSocial",
)

47 runs below 4000000 steps, 8 already at or past it
  steps to add: 999,749 to 1,100,113 each, 47,097,043 total


In [10]:
reached_rest = tsf.resume_training_runs(
    run_ids=REST,
    results_root=RESULTS_ROOT,
    run_dir=RUN_DIR,
    in_yaml=IN_YAML,
    unity_build=UNITY_BUILD,
    max_steps=MAX_STEPS,
    extended_yaml=EXTENDED_YAML,   # the same SocialConfig_4e06.yaml the entrants used
    n_envs=N_ENVS,
    base_port=BASE_PORT,           # 5005 + 20*i -> 5005..5941 for 47 runs
    seed=17,
    dry_run=False,                  # set False to launch
)

extended config written: /home/tom/repos/agent_training/configs/SocialConfig_4e06.yaml  (max_steps -> 4000000)

Resuming 47 runs under /home/tom/Unity/Octagon/results/2607/260723
  260723_0002: 3,000,086 ->   4,000,000  (+999,914 steps)  run-id '2607/260723/260723_0002'
  260723_0003: 3,000,048 ->   4,000,000  (+999,952 steps)  run-id '2607/260723/260723_0003'
  260723_0004: 3,000,004 ->   4,000,000  (+999,996 steps)  run-id '2607/260723/260723_0004'
  260723_0005: 3,000,001 ->   4,000,000  (+999,999 steps)  run-id '2607/260723/260723_0005'
  260723_0006: 3,000,008 ->   4,000,000  (+999,992 steps)  run-id '2607/260723/260723_0006'
  260723_0007: 3,000,077 ->   4,000,000  (+999,923 steps)  run-id '2607/260723/260723_0007'
  260723_0008: 3,000,020 ->   4,000,000  (+999,980 steps)  run-id '2607/260723/260723_0008'
  260723_0009: 3,000,238 ->   4,000,000  (+999,762 steps)  run-id '2607/260723/260723_0009'
  260723_0010: 3,000,094 ->   4,000,000  (+999,906 steps)  run-id '2607/260723/260723